# EZStats - Train Player / Goalkeeper / Referee / Ball Detector  (#1 - HIGHEST ROI)

**Why:** the current detector is `yolov8n` (smallest model). Missed/flickery detections are the #1 cause of tracking ID swaps and missed events. Everything downstream depends on this.

**This run = YOUR OWN data (the same 298-image set), BETTER training** (so it is not the bit-identical wasted re-run):

| | Old model | This notebook |
|---|---|---|
| Data | your `detector/` (298 imgs) | **your `detector/` (298 imgs) - uploaded by you** |
| Model size | yolov8**n** | yolov8**m** |
| Epochs | ~50 | **150 + early stop** |
| Augmentation | tutorial default | **mosaic + mixup + scale + HSV + flip** (expands a small dataset) |
| LR schedule | linear | **cosine** |
| Resolution | 1280 | 1280 |

**Class order (MUST stay):** `0=ball, 1=goalkeeper, 2=player, 3=referee`

---
### BEFORE you start - upload your data (one time)
1. On your laptop, `detector.zip` is on your **Desktop** (Claude made it).
2. Go to https://drive.google.com → open your `ezstats` folder.
3. Upload `detector.zip` anywhere inside `ezstats/` (cell D auto-finds it).
4. Upload THIS notebook into `ezstats/` and open it in Colab.

### Run order: A -> B -> C -> D -> E -> F -> G. Run each cell top to bottom.

## A - Turn on the GPU (do this FIRST)
1. Top menu: **Runtime -> Change runtime type**
2. Hardware accelerator -> **T4 GPU** (free) is fine. A100 (Colab Pro) is much faster.
3. Click **Save**, then run the cell below. You should see a green table.

In [ ]:
!nvidia-smi

## B - Connect Google Drive (your data is read from here; model saves here too)
A popup asks you to pick your account and click **Allow**. We train **directly onto Drive** so a Colab disconnect never loses progress - you just re-run cell E and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
HOME = os.getcwd()
RUNS_DIR = Path('/content/drive/MyDrive/ezstats/runs')   # model saves here
RUN_NAME = 'player_detector_v2'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Working dir :', HOME)
print('Saving to   :', RUNS_DIR / RUN_NAME)

## C - Install the training library

In [ ]:
!pip install -q ultralytics

## D - Unzip YOUR uploaded data + build a clean data.yaml
Auto-finds `detector.zip` anywhere under `MyDrive`. The extractor normalizes Windows backslash paths (PowerShell zips use `\`, which Colab would otherwise flatten into the root). You should see `train images: 298  val images: 49` at the end - if either is 0, the zip is wrong.

In [ ]:
import zipfile, glob, shutil
from pathlib import Path

# Auto-find detector.zip wherever you dropped it under MyDrive
_cands = sorted(glob.glob('/content/drive/MyDrive/**/detector.zip', recursive=True))
assert _cands, 'detector.zip not found in MyDrive -> upload it into your ezstats folder first'
DATA_ZIP = Path(_cands[0])
print('Using data zip:', DATA_ZIP)

DATA_DIR = Path('/content/datasets/detector')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Normalize Windows backslash entry names -> real folders
with zipfile.ZipFile(DATA_ZIP) as z:
    for info in z.infolist():
        name = info.filename.replace('\\', '/')
        if name.endswith('/'):
            continue
        tgt = DATA_DIR / name
        tgt.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as s, open(tgt, 'wb') as d:
            shutil.copyfileobj(s, d)

# Write a clean data.yaml with absolute paths + locked class order
DATA_YAML = DATA_DIR / 'data.yaml'
DATA_YAML.write_text(
    f'train: {DATA_DIR}/train/images\n'
    f'val: {DATA_DIR}/valid/images\n'
    f'test: {DATA_DIR}/test/images\n'
    'nc: 4\n'
    "names: ['ball', 'goalkeeper', 'player', 'referee']\n"
)
print(DATA_YAML.read_text())
n_train = len(list((DATA_DIR / 'train' / 'images').glob('*')))
n_val   = len(list((DATA_DIR / 'valid' / 'images').glob('*')))
print(f'train images: {n_train}   val images: {n_val}')
assert n_train > 0 and n_val > 0, 'Extraction produced 0 images - check the zip structure'

## E - Train (your data, better training)
150 epochs with `patience=40` (stops early if it plateaus, so 'more epochs' can't overfit). Strong augmentation makes the 298-image set behave like a bigger one.

**Time:** ~3-5 h on free T4 (small dataset), ~40 min on A100. For a quick first look, change `epochs=150` to `epochs=60`.

**If Colab disconnects:** just re-run this cell - it auto-resumes from the last checkpoint on Drive.

In [ ]:
%cd {HOME}

ckpt = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
if ckpt.exists():
    print('Found checkpoint -> RESUMING from', ckpt)
    !yolo task=detect mode=train resume=True model='{ckpt}'
else:
    print('Fresh training run.')
    !yolo task=detect mode=train \
      model=yolov8m.pt \
      data='{DATA_YAML}' \
      epochs=150 \
      imgsz=1280 \
      batch=8 \
      patience=40 \
      cos_lr=True \
      close_mosaic=15 \
      hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 \
      degrees=0.0 translate=0.1 scale=0.5 fliplr=0.5 \
      mosaic=1.0 mixup=0.1 \
      plots=True \
      project='{RUNS_DIR}' name='{RUN_NAME}'

## F - Check the results (look before you trust)
Target: **mAP50 > ~0.85** and **recall (R) for `player` > ~0.85**. Low player-recall = still missing players = will NOT fix tracking.

In [ ]:
%cd {HOME}
!yolo task=detect mode=val \
  model='{RUNS_DIR}/{RUN_NAME}/weights/best.pt' \
  data='{DATA_YAML}' imgsz=1280

In [ ]:
from IPython.display import Image
Image(filename=f'{RUNS_DIR}/{RUN_NAME}/results.png', width=900)

In [ ]:
Image(filename=f'{RUNS_DIR}/{RUN_NAME}/val_batch0_pred.jpg', width=900)

## G - Done - the model is already on your Drive
`MyDrive/ezstats/runs/player_detector_v2/weights/best.pt`

**On your laptop (do NOT touch pipeline code yet):**
1. Download `best.pt` from Drive.
2. Save it to `artifacts/training/player_detector_v2/weights/best.pt` (NEW path - keep the old model as fallback).
3. Tell Claude. We validate it side-by-side on `08fd33_4.mp4` and only swap the default if it visibly wins.